##IMAGE SETS GENERATION: TRAIN:VALIDATION:TEST :: 80:10:10

In [ ]:
import os
import random

# ============================================================
# PATHS (EDIT THESE)
# ============================================================

image_dir = "/content/drive/MyDrive/mes_project/KITTI_AUG/image_2"     # merged images
label_dir = "/content/drive/MyDrive/mes_project/KITTI_AUG/label_2"
velodyne_dir = "/content/drive/MyDrive/mes_project/KITTI_AUG/velodyne"
calib_dir = "/content/drive/MyDrive/mes_project/KITTI_AUG/calib"

output_dir = "/content/drive/MyDrive/mes_project/KITTI_AUG/ImageSets"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# GET FILE IDS
# ============================================================

files = []

for f in os.listdir(image_dir):
    if f.endswith((".png",".jpg",".jpeg")):
        files.append(f.split(".")[0])

print("Total files:", len(files))

# shuffle
random.shuffle(files)

# ============================================================
# SPLIT (80-10-10)
# ============================================================

n = len(files)

train_split = int(0.8 * n)
val_split = int(0.9 * n)

train_files = files[:train_split]
val_files = files[train_split:val_split]
test_files = files[val_split:]

print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

# ============================================================
# SAVE FILES
# ============================================================

def save_split(file_list, filename):
    with open(os.path.join(output_dir, filename), "w") as f:
        for item in file_list:
            f.write(item + "\n")

save_split(train_files, "train.txt")
save_split(val_files, "val.txt")
save_split(test_files, "test.txt")

print("ImageSets created successfully!")

Total files: 3354
Train: 2683
Val: 335
Test: 336
ImageSets created successfully!


##INFOS FILE GENERATION

In [ ]:
!pip install tqdm -q

import os
import pickle
import numpy as np
from tqdm import tqdm


root = "/content/drive/MyDrive/mes_project/KITTI_AUG"

image_dir = os.path.join(root, "image_2")
velodyne_dir = os.path.join(root, "velodyne")
label_dir = os.path.join(root, "label_2")
calib_dir = os.path.join(root, "calib")
imageset_dir = os.path.join(root, "ImageSets")

# ============================================================
# READ CALIB FILE
# ============================================================

def read_calib(calib_path):

    calib = {}

    with open(calib_path) as f:
        for line in f:
            key, value = line.split(":",1)
            values = np.array([float(x) for x in value.strip().split()])

            if key == "P2":
                calib[key] = values.reshape(3,4)
            elif key == "R0_rect":
                calib[key] = values.reshape(3,3)
            elif key == "Tr_velo_to_cam":
                calib[key] = values.reshape(3,4)
            else:
                calib[key] = values

    return calib

# ============================================================
# READ 3D LABEL FILE (KITTI FORMAT)
# ============================================================

def read_label(label_path):

    objects = []

    if not os.path.exists(label_path):
        return objects

    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:

        parts = line.strip().split()

        if len(parts) < 15:
            continue

        obj = {
            "name": parts[0],

            # 2D bbox
            "bbox": list(map(float, parts[4:8])),

            # 3D dimensions (h, w, l)
            "dimensions": list(map(float, parts[8:11])),

            # 3D location (x, y, z)
            "location": list(map(float, parts[11:14])),

            # rotation
            "rotation_y": float(parts[14])
        }

        objects.append(obj)

    return objects

# ============================================================
# CREATE INFO FILE FUNCTION
# ============================================================

def create_info_file(split_name):

    split_file = os.path.join(imageset_dir, split_name + ".txt")

    with open(split_file) as f:
        ids = [x.strip() for x in f.readlines()]

    infos = []

    for idx in tqdm(ids):

        info = {}

        # paths
        image_path = os.path.join(image_dir, idx + ".png")
        velodyne_path = os.path.join(velodyne_dir, idx + ".bin")
        label_path = os.path.join(label_dir, idx + ".txt")
        calib_path = os.path.join(calib_dir, idx + ".txt")

        # basic info
        info["point_cloud"] = {
            "num_features": 4,
            "lidar_path": velodyne_path
        }

        info["image"] = {
            "image_idx": idx,
            "image_path": image_path
        }

        # calibration
        info["calib"] = read_calib(calib_path)

        # annotations
        objects = read_label(label_path)

        if len(objects) > 0:

            names = []
            bboxes = []
            dims = []
            locs = []
            rots = []

            for obj in objects:
                names.append(obj["name"])
                bboxes.append(obj["bbox"])
                dims.append(obj["dimensions"])
                locs.append(obj["location"])
                rots.append(obj["rotation_y"])

            info["annos"] = {
                "name": np.array(names),
                "bbox": np.array(bboxes),
                "dimensions": np.array(dims),
                "location": np.array(locs),
                "rotation_y": np.array(rots)
            }

        infos.append(info)

    return infos

# ============================================================
# GENERATE INFO FILES
# ============================================================

print("Creating TRAIN info...")
train_infos = create_info_file("train")

print("Creating VAL info...")
val_infos = create_info_file("val")

print("Creating TEST info...")
test_infos = create_info_file("test")

# save
with open(os.path.join(root, "train_infos.pkl"), "wb") as f:
    pickle.dump(train_infos, f)

with open(os.path.join(root, "val_infos.pkl"), "wb") as f:
    pickle.dump(val_infos, f)

with open(os.path.join(root, "test_infos.pkl"), "wb") as f:
    pickle.dump(test_infos, f)

print("\nINFO FILES GENERATED SUCCESSFULLY 🚀")

Creating TRAIN info...


100%|██████████| 2683/2683 [54:59<00:00,  1.23s/it]


Creating VAL info...


100%|██████████| 335/335 [06:49<00:00,  1.22s/it]


Creating TEST info...


100%|██████████| 336/336 [06:43<00:00,  1.20s/it]


INFO FILES GENERATED SUCCESSFULLY 🚀


##GT DATABASE SCRIPT

In [3]:
# ============================================================
# INSTALL
# ============================================================
!pip install tqdm -q

import os
import pickle
import numpy as np
import time
from tqdm import tqdm

# ============================================================
# PATH
# ============================================================

root = "/content/drive/MyDrive/mes_project/KITTI_AUG"   # change to KITTI_AUG later

velodyne_dir = os.path.join(root, "velodyne")
info_path = os.path.join(root, "train_infos.pkl")

db_save_path = os.path.join(root, "gt_database")
db_info_save_path = os.path.join(root, "kitti_dbinfos_train.pkl")

os.makedirs(db_save_path, exist_ok=True)

# ============================================================
# LOAD INFOS
# ============================================================

with open(info_path, "rb") as f:
    infos = pickle.load(f)

print("Total training samples:", len(infos))

# ============================================================
# SIMPLE OBJECT EXTRACTION (KEY FIX)
# ============================================================

def extract_object_points(points, center, radius=3.0):
    """
    Extract points around object center (robust method)
    """

    cx, cy, cz = center

    # distance-based filtering
    dist = np.sqrt(
        (points[:,0] - cx)**2 +
        (points[:,1] - cy)**2 +
        (points[:,2] - cz)**2
    )

    obj_pts = points[dist < radius]

    # fallback if empty
    if len(obj_pts) < 20:
        idx = np.random.choice(len(points), size=min(200, len(points)), replace=False)
        obj_pts = points[idx]

    return obj_pts

# ============================================================
# MAIN LOOP
# ============================================================

all_db_infos = {}

start_time = time.time()
total_objects = 0

for idx_i, info in enumerate(tqdm(infos, desc="GT DB Creation")):

    idx = info["image"]["image_idx"]
    velodyne_path = info["point_cloud"]["lidar_path"]

    if not os.path.exists(velodyne_path):
        continue

    # load lidar
    points = np.fromfile(velodyne_path, dtype=np.float32).reshape(-1,4)

    if "annos" not in info:
        continue

    annos = info["annos"]

    names = annos["name"]
    locs = annos["location"]

    # IMPORTANT: use location directly (no transform)
    for i in range(len(names)):

        name = str(names[i])
        center = locs[i]

        obj_points = extract_object_points(points[:,:3], center)

        if obj_points.shape[0] < 10:
            continue

        filename = f"{name}_{idx}_{i}.bin"
        filepath = os.path.join(db_save_path, filename)

        obj_points.tofile(filepath)

        db_info = {
            "name": name,
            "path": filepath,
            "image_idx": idx,
            "gt_idx": i,
            "num_points_in_gt": obj_points.shape[0]
        }

        if name not in all_db_infos:
            all_db_infos[name] = []

        all_db_infos[name].append(db_info)

        total_objects += 1

    # ========================================================
    # LOGGING
    # ========================================================
    if idx_i % 100 == 0 and idx_i > 0:

        elapsed = time.time() - start_time
        speed = idx_i / elapsed

        print(f"\nProcessed {idx_i}/{len(infos)}")
        print(f"Speed: {speed:.2f} images/sec")
        print(f"Objects so far: {total_objects}")

# ============================================================
# SAVE
# ============================================================

with open(db_info_save_path, "wb") as f:
    pickle.dump(all_db_infos, f)

total_time = time.time() - start_time

print("\n===================================")
print("GT DATABASE CREATED SUCCESSFULLY 🚀")
print("===================================")
print(f"Total objects extracted: {total_objects}")
print(f"Total time: {total_time/60:.2f} minutes")
print("===================================")

Total training samples: 2683


GT DB Creation:   4%|▍         | 102/2683 [00:12<05:21,  8.03it/s]


Processed 100/2683
Speed: 7.88 images/sec
Objects so far: 539


GT DB Creation:   7%|▋         | 201/2683 [00:33<04:48,  8.61it/s]


Processed 200/2683
Speed: 5.96 images/sec
Objects so far: 1035


...


Processed 2400/2683
Speed: 6.14 images/sec
Objects so far: 13554


GT DB Creation:  97%|█████████▋| 2602/2683 [07:13<00:15,  5.36it/s]


Processed 2600/2683
Speed: 6.00 images/sec
Objects so far: 14623


GT DB Creation: 100%|██████████| 2683/2683 [07:29<00:00,  5.98it/s]


GT DATABASE CREATED SUCCESSFULLY 🚀
Total objects extracted: 15097
Total time: 7.49 minutes
